## Funciones: Data, spark y demás

- Rutas, sesión de Spark y escritura de resultados

- Compartidas por todos los notebooks (`%run ../funciones.ipynb`)

- Salidas van a `<carpeta>/temp/<NOTEBOOK>/`

### Imports

In [ ]:
import os, socket, time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, Window, functions as F

%matplotlib inline
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})
pd.set_option("display.max_colwidth", 80)

### Rutas

`raw` → `bronze` → `staging` → `final` → `artefactos`

In [ ]:
proyecto = Path.cwd()
while not (proyecto / "docker-compose.yml").exists() and proyecto != proyecto.parent:
    proyecto = proyecto.parent

raw_dir     = proyecto / "data" / "raw"
bronze_dir  = proyecto / "data" / "bronze"
staging_dir = proyecto / "data" / "staging"
final_dir   = proyecto / "data" / "final"
artef_dir   = proyecto / "data" / "artefactos"
tc_path     = proyecto / "data" / "cambios.csv"
anios = [2023, 2024, 2025]

out_dir = Path.cwd() / "temp" / NOTEBOOK
fig_dir, tab_dir, web_dir = out_dir / "figs", out_dir / "tablas", out_dir / "web"
for d in (fig_dir, tab_dir, web_dir):
    d.mkdir(parents=True, exist_ok=True)

### Spark

- Master del cluster (`spark://spark-master:7077`)

- Espera a que los executors se registren antes de seguir

In [ ]:
def crear_spark(nombre, particiones=24, espera=90):
    master = os.environ.get("spark_master", "local[*]")
    b = (SparkSession.builder.appName(nombre).master(master)
         .config("spark.driver.memory",   os.environ.get("SPARK_DRIVER_MEMORY", "2g"))
         .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY", "2g"))
         .config("spark.sql.shuffle.partitions", str(particiones))
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.sql.optimizer.canChangeCachedPlanOutputPartitioning", "false"))
    if master.startswith("spark://"):
        b = b.config("spark.driver.host", socket.gethostname()).config("spark.driver.bindAddress", "0.0.0.0")
    s = b.getOrCreate()
    s.sparkContext.setLogLevel("ERROR")
    t0 = time.time()
    while master.startswith("spark://") and n_executors(s) == 0 and time.time() - t0 < espera:
        time.sleep(2)
    return s

def n_executors(s):
    return s.sparkContext._jsc.sc().getExecutorMemoryStatus().size() - 1

def verificar_cluster(s):
    sc = s.sparkContext
    print(f"Master        : {sc.master}")
    print(f"Executors     : {n_executors(s)}")
    print(f"Cores totales : {sc.defaultParallelism}")
    print(f"Memoria driver: {sc._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3:.1f} GB")
    print("UI del job    : http://localhost:4040")
    if sc.master.startswith("spark://"):
        assert n_executors(s) > 0, "No hay executors registrados: revisar `docker compose ps`"

### Lectura y escritura

- `guardar_tabla`: parquet + json, y devuelve las filas escritas

- `verificar_salidas`: relee cada tabla y compara filas

In [ ]:
def leer(directorio, nombre):
    return SparkSession.getActiveSession().read.parquet(str(directorio / nombre))

def mostrar_fig(nombre):
    plt.tight_layout()
    plt.savefig(fig_dir / f"{nombre}.png", dpi=150, bbox_inches="tight")
    plt.show()

def guardar_tabla(df, nombre):
    s = SparkSession.getActiveSession()
    sdf = s.createDataFrame(df) if isinstance(df, pd.DataFrame) else df
    sdf.coalesce(1).write.mode("overwrite").parquet(str(tab_dir / nombre))
    sdf.toPandas().to_json(web_dir / f"{nombre}.json", orient="records", force_ascii=False, date_format="iso", indent=1)
    return len(df) if isinstance(df, pd.DataFrame) else sdf.count()

def guardar_tablas(tablas):
    escritos = {n: guardar_tabla(df, n) for n, df in tablas.items()}
    print(f"Escrito en {out_dir.relative_to(proyecto)}/: {', '.join(escritos)}")
    return escritos

def verificar_salidas(escritos):
    for nombre, n in escritos.items():
        leido = leer(tab_dir, nombre).count()
        assert leido == n, f"{nombre}: esperaba {n}, leí {leido}"
        assert (web_dir / f"{nombre}.json").exists(), f"falta web/{nombre}.json"
    figs = sorted(p.stem for p in fig_dir.glob("*.png"))
    print(f"OK: {len(escritos)} tablas (parquet + json)" + (f" | {len(figs)} figuras: {', '.join(figs)}" if figs else ""))